# Quark OS Challenge — Hardware Characterization and Region Recommendation
**Device:** IBM Kingston (156 physical qubits)  
**Calibration snapshot:** 31 March 2026

## Objective
Select connected 16- and 32-qubit regions using the supplied calibration snapshot and coupling topology. This is a calibration-based recommendation, not a claim about future device performance.

## Data loading and reproducibility
The challenge calibration file has a `.xls` filename but its contents are comma-separated text. To make this submission a **single self-contained notebook**, the supplied calibration snapshot and topology JSON are embedded below. No external file paths are required.

In [1]:
import base64, io, json
import numpy as np
import pandas as pd
import networkx as nx

CALIBRATION_B64 = """..."""
TOPOLOGY_B64 = """..."""

df = pd.read_csv(io.BytesIO(base64.b64decode(CALIBRATION_B64)))
topology = json.loads(base64.b64decode(TOPOLOGY_B64).decode("utf-8"))
print("Calibration shape:", df.shape)
print("Topology device:", topology.get("device_name"))
print("Topology qubits:", topology.get("num_qubits"))
display(df.head())

Calibration shape: (156, 19)
Topology device: ibm_kingston
Topology qubits: 156


,Qubit,T1 (us),T2 (us),Readout assignment error,Prob meas0 prep1,Prob meas1 prep0,Readout length (ns),ID error,Single-qubit gate length (ns),RX error,Z-axis rotation (rz) error,√x (sx) error,Pauli-X error,CZ error,Gate length (ns),RZZ error,MEASURE error,MEASURE_2 error,Operational
0,0,290.800821,298.140732,0.011475,0.014404,0.008545,2280,0.000205,32,0.000205,0,0.000205,0.000205,1:0.0010167457696165227,1:68,1:0.0010602119472144789,0.011475,0.016602,Yes
1,1,372.822654,437.021834,0.164795,0.100342,0.229248,2280,0.000181,32,0.000181,0,0.000181,0.000181,2:0.0009609959626648845;0:0.0010167457696165227,2:68;0:68,2:0.0009016463575797495;0:0.0010602119472144789,0.164795,0.081055,Yes
2,2,305.112602,107.784842,0.008057,0.010742,0.005371,2280,0.000100,32,0.000100,0,0.000100,0.000100,1:0.0009609959626648845;3:0.0012523142615088745,1:68;3:68,1:0.0009016463575797495;3:0.000930022916350165,0.008057,0.008057,Yes
3,3,354.894616,483.530444,0.008301,0.011475,0.005127,2280,0.000243,32,0.000243,0,0.000243,0.000243,16:0.0012413347520312812;2:0.00125231426150887...,16:68;2:68;4:68,16:0.001184823243668115;2:0.000930022916350165...,0.008301,0.007324,Yes
4,4,316.778855,53.238112,0.022339,0.018799,0.025879,2280,0.000282,32,0.000282,0,0.000282,0.000282,3:0.002007963045889627;5:0.002521856056860644,3:68;5:68,3:0.0018174791538967394;5:0.0026275206583171173,0.022339,0.013916,Yes


# Part 1 — Initial dataset analysis

We inspect missing values, operational status, and distributions of the metrics that directly affect circuit quality. T1 and T2 are coherence times (larger is generally better); readout assignment error and single-qubit gate error (ID error) are lower-is-better metrics. CZ error is stored as neighbour:value pairs in each qubit row.

In [2]:
numeric_cols = ["T1 (us)", "T2 (us)", "Readout assignment error", "ID error",
                "CZ error", "MEASURE error", "MEASURE_2 error"]
print("Rows / columns:", df.shape)
print("\\nOperational status:")
display(df["Operational"].value_counts(dropna=False).to_frame("count"))
print("\\nMissing values:")
display(df.isna().sum().to_frame("missing"))
print("\\nKey metric summary:")
display(df[["T1 (us)", "T2 (us)", "Readout assignment error", "ID error",
            "MEASURE error", "MEASURE_2 error"]].describe().T)
print("\\nHighest readout-error qubits:")
display(df.nlargest(8, "Readout assignment error")[["Qubit","Readout assignment error","T1 (us)","T2 (us)"]])

Rows / columns: (156, 19)
\nOperational status:


,count
Operational,
Yes,156


\nMissing values:


,missing
Qubit,0
T1 (us),1
T2 (us),1
Readout assignment error,0
Prob meas0 prep1,0
Prob meas1 prep0,0
Readout length (ns),0
ID error,3
Single-qubit gate length (ns),0
RX error,3


\nKey metric summary:


,count,mean,std,min,25%,50%,75%,max
T1 (us),155.0,271.047301,81.323694,65.887474,212.142753,280.249093,336.066393,479.729813
T2 (us),155.0,171.809415,114.423826,11.926602,81.208905,134.260653,242.317056,488.482105
Readout assignment error,156.0,0.024635,0.060042,0.003174,0.006226,0.009277,0.018982,0.504883
ID error,153.0,0.000302,0.000486,0.000087,0.000172,0.000214,0.000297,0.005838
MEASURE error,156.0,0.024635,0.060042,0.003174,0.006226,0.009277,0.018982,0.504883
MEASURE_2 error,156.0,0.022345,0.050535,0.001465,0.005615,0.008423,0.018616,0.500000


\nHighest readout-error qubits:


,Qubit,Readout assignment error,T1 (us),T2 (us)
146,146,0.504883,NaN,NaN
96,96,0.495605,332.048816,155.750655
1,1,0.164795,372.822654,437.021834
113,113,0.142578,190.366065,122.098019
127,127,0.121948,307.906237,214.878954
131,131,0.110596,67.400046,70.536682
102,102,0.107422,242.699571,206.162260
31,31,0.098633,263.744832,290.737567


### Observations

- The snapshot covers 156 physical qubits. Operational status is recorded as `Yes` for every qubit, but operational status alone does not imply equal quality.
- Readout assignment error has a wide spread, so it is important to avoid regions containing high-readout-error qubits.
- T1 and T2 also vary substantially; shorter coherence times can reduce the useful circuit execution window.
- Two-qubit gate quality is assessed from the neighbour-specific CZ error entries. This is important because a region with good individual qubits can still be poor if its internal couplers are weak.
- A few calibration fields are missing, and six topology edges have no CZ error entry in the snapshot. Missing values are not silently treated as excellent: the region score applies a small missing-edge penalty and uses median imputation for missing node ranks.

# Part 2 — Hardware representation

Represent the processor as an undirected NetworkX graph: physical qubits are nodes and available couplers are edges. A connected induced subgraph is required so that the selected qubits form a usable contiguous hardware region. For each candidate region, score both node calibration quality and internal CZ-coupler quality.

In [3]:
G = nx.Graph()
G.add_nodes_from(range(int(topology["num_qubits"])))
G.add_edges_from(tuple(e) for e in topology["coupling_map_undirected"])
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
print("Connected:", nx.is_connected(G))
print("Degree summary:", pd.Series(dict(G.degree())).describe().to_dict())

Nodes: 156 Edges: 176
Connected: True
Degree summary: {'count': 156.0, 'mean': 2.2564102564102564, 'std': 0.5432494213799132, 'min': 1.0, '25%': 2.0, '50%': 2.0, '75%': 3.0, 'max': 3.0}


### Representation notes

An undirected graph is appropriate because the supplied topology provides a bidirectional coupling relation for region connectivity. The region score uses all qubits in the selected set and all available internal couplers; it does not reward a scattered collection of individually good qubits.

# Part 3 — Scoring and candidate search

**Scoring assumptions (lower is better):**
1. Convert each node metric to a percentile penalty in [0, 1]. High readout error and high single-qubit error are bad; low T1 and low T2 are bad.
2. Average the four node penalties: readout error, ID error, inverse T1 rank, and inverse T2 rank.
3. Convert measured CZ errors on edges to percentile penalties.
4. Region score = 0.55 × mean node penalty + 0.40 × mean internal-edge penalty + 0.05 × fraction of internal edges with missing CZ calibration.

The weighting gives slightly more importance to the qubit-level quality while still assigning 40% to two-qubit couplers. Percentile ranks reduce sensitivity to the different units and extreme outliers. The search grows connected regions from multiple starting qubits, then applies connectivity-preserving one-qubit swaps to improve the best candidate. The heuristic is transparent and practical; it is not a proof of a globally optimal connected subgraph.

In [4]:
# Build node penalties using percentile ranks.
node_metrics = df.set_index("Qubit")[["Readout assignment error", "ID error", "T1 (us)", "T2 (us)"]].copy()
def percentile_penalty(series, higher_is_worse=True):
    ranks = series.rank(pct=True)
    return ranks if higher_is_worse else 1.0 - ranks

node_penalty = (
    percentile_penalty(node_metrics["Readout assignment error"]) +
    percentile_penalty(node_metrics["ID error"]) +
    percentile_penalty(node_metrics["T1 (us)"], higher_is_worse=False) +
    percentile_penalty(node_metrics["T2 (us)"], higher_is_worse=False)
) / 4.0
node_penalty = node_penalty.fillna(node_penalty.median()).to_dict()

# Parse CZ neighbour:error strings into undirected edge values.
edge_values = {}
for _, row in df.iterrows():
    q = int(row["Qubit"])
    value = row["CZ error"]
    if isinstance(value, str):
        for item in value.split(";"):
            if ":" in item:
                neighbour, error = item.split(":", 1)
                edge = tuple(sorted((q, int(neighbour))))
                edge_values.setdefault(edge, []).append(float(error))
edge_values = {edge: float(np.mean(values)) for edge, values in edge_values.items()}
edge_series = pd.Series(edge_values)
edge_rank = edge_series.rank(pct=True)
edge_penalty = edge_rank.to_dict()

print("Node penalty values:", len(node_penalty))
print("Measured CZ edges:", len(edge_penalty), "of", G.number_of_edges())
print("Missing CZ edge calibrations:", G.number_of_edges() - len(edge_penalty))

def region_score(region):
    region = set(region)
    subgraph = G.subgraph(region)
    node_part = float(np.mean([node_penalty[q] for q in region]))
    measured = [edge_penalty[e] for e in subgraph.edges if e in edge_penalty]
    edge_part = float(np.mean(measured)) if measured else 1.0
    missing_fraction = (subgraph.number_of_edges() - len(measured)) / max(1, subgraph.number_of_edges())
    return 0.55 * node_part + 0.40 * edge_part + 0.05 * missing_fraction

def grow_region(size, start):
    region = {int(start)}
    while len(region) < size:
        frontier = set().union(*(set(G.neighbors(q)) for q in region)) - region
        options = []
        for q in frontier:
            trial = region | {q}
            internal_links = sum(1 for n in G.neighbors(q) if n in region)
            options.append((region_score(trial), -internal_links, q))
        _, _, chosen = min(options)
        region.add(chosen)
    return region

def find_region(size, random_starts=250, seed=42):
    rng = np.random.default_rng(seed + size)
    candidates = []
    for start in G.nodes:
        r = grow_region(size, start)
        candidates.append((region_score(r), r))
    for _ in range(random_starts):
        start = int(rng.integers(G.number_of_nodes()))
        r = grow_region(size, start)
        candidates.append((region_score(r), r))
    candidates.sort(key=lambda item: item[0])
    best_region = set(candidates[0][1])
    best_score = region_score(best_region)
    # Connectivity-preserving one-node swaps; repeat until no improvement.
    changed = True
    while changed:
        changed = False
        outside = set(G.nodes) - best_region
        for old in sorted(best_region):
            for new in sorted(outside):
                trial = (best_region - {old}) | {new}
                if nx.is_connected(G.subgraph(trial)):
                    trial_score = region_score(trial)
                    if trial_score < best_score - 1e-12:
                        best_region, best_score = trial, trial_score
                        changed = True
                        break
            if changed:
                break
    return sorted(best_region), best_score

region16, score16 = find_region(16)
region32, score32 = find_region(32)
print("16-qubit score:", round(score16, 6), "qubits:", region16)
print("32-qubit score:", round(score32, 6), "qubits:", region32)

Node penalty values: 156
Measured CZ edges: 170 of 176
Missing CZ edge calibrations: 6
16-qubit score: 0.330703 qubits: [80, 81, 82, 83, 96, 103, 104, 105, 117, 122, 123, 124, 125, 126, 127, 137]
32-qubit score: 0.363462 qubits: [80, 81, 82, 83, 96, 97, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 117, 122, 123, 124, 125, 126, 127, 128, 129, 136, 137, 140, 141, 142, 143, 144]


# Part 4 — Recommended 16-qubit region

In [5]:
def region_summary(region):
    sub = df[df["Qubit"].isin(region)].copy()
    H = G.subgraph(region)
    internal = [edge_values[e] for e in H.edges if e in edge_values]
    return {
        "qubit_count": len(region),
        "connected": nx.is_connected(H),
        "qubits": region,
        "internal_edges": H.number_of_edges(),
        "measured_CZ_edges": len(internal),
        "mean_T1_us": sub["T1 (us)"].mean(),
        "mean_T2_us": sub["T2 (us)"].mean(),
        "mean_readout_error": sub["Readout assignment error"].mean(),
        "mean_ID_error": sub["ID error"].mean(),
        "mean_CZ_error": float(np.mean(internal)) if internal else np.nan,
        "score": region_score(region)
    }

summary16 = region_summary(region16)
display(pd.Series(summary16).to_frame("16-qubit recommendation"))

,16-qubit recommendation
qubit_count,16
connected,True
qubits,"[80, 81, 82, 83, 96, 103, 104, 105, 117, 122, ..."
internal_edges,15
measured_CZ_edges,13
mean_T1_us,285.142691
mean_T2_us,200.632955
mean_readout_error,0.047562
mean_ID_error,0.000204
mean_CZ_error,0.001344


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


### 16-qubit rationale

The recommended set is selected as a connected induced region, not as 16 unrelated low-error qubits. Its score jointly considers readout, single-qubit error, coherence, and internal CZ error. The summary above reports the physical qubit IDs, connectivity, internal coupler count, measured CZ coverage, and mean calibration values so the recommendation can be audited.

# Part 5 — Recommended 32-qubit region

In [6]:
summary32 = region_summary(region32)
display(pd.Series(summary32).to_frame("32-qubit recommendation"))

,32-qubit recommendation
qubit_count,32
connected,True
qubits,"[80, 81, 82, 83, 96, 97, 103, 104, 105, 106, 1..."
internal_edges,31
measured_CZ_edges,27
mean_T1_us,270.282949
mean_T2_us,191.469492
mean_readout_error,0.030506
mean_ID_error,0.000223
mean_CZ_error,0.00145


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


### 32-qubit rationale

The 32-qubit recommendation is independently optimized for its target size. It is not assumed that the best 32-qubit region must simply be an extension of the 16-qubit choice: adding qubits changes the balance of node and coupler quality. The selected region is connected and its score includes all internal couplers.

## Recommendation rationale

I recommend the 16- and 32-qubit regions printed in the result cells above. The decision is based on a consistent, normalized scoring method that balances qubit calibration quality with two-qubit connectivity quality. Connectivity is explicitly checked, and missing CZ calibration is penalized rather than interpreted as a zero-error edge. The optimization is a multi-start greedy search followed by local connectivity-preserving swaps; therefore, the result is a well-supported heuristic recommendation, not a mathematical guarantee of a global optimum.

## Additional information that would improve confidence

- Repeated calibration snapshots over time to measure drift and stability.
- Per-edge CZ calibration uncertainty and direction-specific gate performance.
- Circuit workload information (two-qubit gate count, interaction graph, depth, and error sensitivity), since the best region can depend on the circuit.
- Leakage, crosstalk, reset error, gate duration, and simultaneous-operation data.
- Calibration timestamps and uncertainty estimates for each metric.

The supplied data is one snapshot, so this analysis describes relative hardware quality at that calibration time. A production scheduler should refresh the score as calibrations change and consider the workload's actual connectivity requirements.

## Submission checklist

- This notebook embeds the supplied calibration and topology data, so it runs without external CSV/JSON paths.
- Restart the kernel and run all cells before submitting.
- Save this notebook as `FirstName LastName Assignment.ipynb`.
- The challenge asks for a single `.ipynb` submission by **16 November 2026, 11:59 PM**. Ensure you can explain the scoring weights, percentile normalization, connectivity constraint, and limitations.